This notebook will go through each stage in the training pipeline from data prep and model building to training and evaluation.  
The goal is mainly for debugging and understanding what happens at each step.

In [1]:
# First need to add src to path
import sys
import os
import yaml
from pathlib import Path

# project_root = Path("..")  # example
project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


In [2]:
from src.data.datasets import SegmentationDataset
from src.data.transforms import get_baseline_train_transforms, get_baseline_val_transforms, get_baseline_test_transforms
from src.data.loader import get_datasets, get_dataloaders

from src.models.model_factory import build_model

from src.training.train import train_one_epoch, evaluate

from src.metrics.segmentation import *



/Users/giladfelsen/my_dl_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config_path = "/Users/giladfelsen/my_dl_project/configs/default.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg

{'data_dir': 'data/raw',
 'dataset_name': 'VOC2012_seg_subset_trainval',
 'image_subdir': 'JPEGImages',
 'mask_subdir': 'SegmentationClass',
 'splits_subdir': 'ImageSets/Segmentation',
 'train_split': 'train.txt',
 'val_split': 'val.txt',
 'val_fraction': 0.5,
 'batch_size': 8,
 'num_workers': 4,
 'pin_memory': False,
 'shuffle_train': True,
 'test_batch_size': 1,
 'image_size': 128,
 'model': {'name': 'resnet34_unet',
  'num_classes': 21,
  'freeze_encoder': False},
 'optimizer': {'lr': 0.001, 'weight_decay': 1e-05},
 'trainer': {'epochs': 1},
 'metrics': {'ignore_index': 255},
 'debug_samples': None}

Pipeline:

`src.training.train.main()` orchestrates everything.
1) Load train/val/test image/mask transforms with `src.data.transforms.py`
2) Load train/val/test dataloaders with `src.data.loader.py` (datasets loaded implcitly). Pass transforms from above as inputs. 
3) Initialize a new model with `src.models.model_factory.py`.
4) Initialize a new optimizer (Adam) in `main()`
5) Initialize Logging (SummaryWriter object) `log_dir="runs/baseline"`. initial `best_val_loss = float('inf')` for checkpointing best model.
6) For n epochs do:  
    6.1) `train_loss = train_one_epoch()` function in `src.training.train.py`.
    6.2) `val_stats = evaluate()`
    6.3) Print stats, write stats to tensorboard with SummaryWriter, checkpoint best model


# Dataset

In [4]:



train_trans = get_baseline_train_transforms()
val_trans = get_baseline_val_transforms()
test_trans = get_baseline_test_transforms()

train_dataset, val_dataset, test_dataset = get_datasets(cfg, train_trans, val_trans, test_trans)
train_loader, val_loader, test_loader = get_dataloaders(config_path, train_trans, val_trans, test_trans)

In [6]:
images, masks = next(iter(train_loader))

In [7]:
torch.unique(masks)

tensor([  0,  34,  52,  75,  76, 128, 147, 220], dtype=torch.uint8)

In [8]:
train_trans

Compose([
  Resize(p=1.0, area_for_downscale=None, height=256, interpolation=1, mask_interpolation=0, width=256),
  Normalize(p=1.0, max_pixel_value=255.0, mean=(0.485, 0.456, 0.406), normalization='standard', std=(0.229, 0.224, 0.225)),
  ToTensorV2(p=1.0, transpose_mask=False),
], p=1.0, bbox_params=None, keypoint_params=None, additional_targets={}, is_check_shapes=True)

In [ ]:
from PIL import Image
im_path = "/Users/giladfelsen/my_dl_project/data/raw/VOC2012_seg_subset_trainval/SegmentationClass/2007_000032.png"
im = Image(open(im_path))